# Telegram Platform Analysis for CDDBS

**Sprint**: 3 — Multi-Platform Support  
**Author**: CDDBS Research  
**Date**: March 2026  
**Purpose**: Research Telegram-specific disinformation patterns, platform architecture, and analysis methodology to extend CDDBS beyond Twitter.

---

## 1. Telegram Platform Architecture

### Entity Types

| Entity Type | Description | Max Members | Admin Visibility | Key for CDDBS |
|------------|-------------|-------------|-----------------|----------------|
| **Channel** | Broadcast-only; only admins post | Unlimited | Hidden by default | Primary target — state media channels |
| **Group** | Multi-user chat; anyone can post | 200 | Visible | Secondary — coordination groups |
| **Supergroup** | Upgraded group with admin tools | 200,000 | Configurable | Secondary — large discussion groups |
| **Bot** | Automated account via Bot API | N/A | N/A | Amplification tool detection |

### Key Differences from Twitter

| Feature | Twitter | Telegram | CDDBS Impact |
|---------|---------|----------|---------------|
| Content visibility | Public by default | Channels public; groups may be private | Limits data access for private groups |
| Engagement metrics | Likes, retweets, replies visible | Views visible; no public likes | Different engagement analysis needed |
| Content amplification | Retweets/quotes | Message forwarding (with source attribution) | Forwarding chains are **more traceable** than retweets |
| Admin anonymity | Account owner visible | Channel admins hidden by default | Attribution is harder |
| Content moderation | Platform-level moderation | Minimal platform moderation | More unfiltered content to analyze |
| Account creation | Email/phone, visible creation date | Phone number, creation date not exposed | Harder to assess account age |
| Threading | Reply threads | Reply-to-message | Different conversation structure |
| Search | Full-text search via API | Limited search; channel-specific | Harder to discover content |
| Media | Images, videos, polls | Images, videos, polls, files, voice | Richer media types to analyze |
| Deletion | Visible to account owner | Can be deleted for all participants | Deletion tracking important |

## 2. Telegram-Specific Disinformation Techniques

### 2.1 Forwarding Chain Manipulation

Telegram's message forwarding retains the source channel, creating traceable amplification chains. Disinformation actors exploit this:

- **Laundering chains**: Content originates on anonymous channel → forwarded through intermediary channels → reaches mainstream channels. Each hop adds perceived legitimacy.
- **Forwarding velocity**: Coordinated networks forward content within seconds, achieving rapid amplification before fact-checking can respond.
- **Forward stripping**: Reposting content without forwarding (copy-paste) to remove source attribution. Detectable via content fingerprinting.

### 2.2 Channel Networks

Russian state-linked Telegram operations use layered channel networks:

```
Tier 1: Official channels (RT English, Sputnik, TASS)
   ↓ forward
Tier 2: Semi-official proxy channels (branded as "independent analysis")
   ↓ forward
Tier 3: Partisan channels (political commentary, "truth" channels)
   ↓ forward / copy-paste
Tier 4: Organic audiences (genuine users who share content)
```

**Key finding**: The tier structure maps directly to the GEC's 5-pillar model:
- Tier 1 = Pillar 2 (State-Funded Global Messaging)
- Tier 2 = Pillar 3 (Proxy Sites)
- Tier 3 = Pillar 4 (Weaponized Social Media)
- Tier 4 = Organic reach

### 2.3 Bot Farms on Telegram

Telegram bots can:
- Auto-forward messages between channels
- Inflate subscriber counts (join/leave cycles)
- Post scheduled content across multiple channels simultaneously
- Aggregate content from multiple sources into a single channel

**Detection signals**:
- Forwarding within <1 second of source post
- 24/7 posting with no gaps
- Identical content across 10+ channels within minutes
- Subscriber growth spikes not correlated with content events

### 2.4 Anonymous Administration

Unlike Twitter, Telegram channel admins are hidden by default. This enables:
- **Deniable operation**: State actors can run channels without visible attribution
- **Network of anonymous channels**: Impossible to prove common ownership through admin data alone
- **Rapid replacement**: If a channel is flagged, operators create new ones instantly

### 2.5 Message Deletion

Telegram allows admins to delete messages retroactively:
- Used to remove content after screenshots are taken ("plausible deniability")
- Used to test narratives: post, measure engagement, delete if unsuccessful
- Detection: Compare message IDs for gaps; track message counts over time

## 3. Telegram-Specific Behavioral Indicators

### Indicators for CDDBS Analysis

| Indicator | Schema Key | Description | Suspicious Threshold |
|-----------|-----------|-------------|---------------------|
| **Forwarding pattern** | `forwarding_pattern` | Ratio of forwarded vs original content; source diversity | >80% forwarded from <3 sources |
| **Channel growth** | `channel_growth` | Subscriber growth rate and pattern | >1000 subscribers/day without viral content |
| **Bot activity** | `bot_activity` | Signs of automated behavior | <1s forwarding delay; 24/7 activity |
| **Message deletion** | `message_deletion` | Frequency and pattern of deleted messages | >10% deletion rate |
| **Posting frequency** | `posting_frequency` | Messages per day; time distribution | Inhuman consistency; no weekday/weekend variation |
| **View-to-subscriber ratio** | `engagement_ratio` | Views per message / subscriber count | <10% may indicate inflated subscribers; >100% may indicate external amplification |
| **Forwarding velocity** | `coordination_signals` | Time between source post and forward | Consistent <5s suggests automation |
| **Content origin diversity** | `content_amplification` | How many different sources does the channel forward from? | <3 unique sources = likely proxy |

### Indicator Confidence Mapping

| Indicator Combination | Confidence Signal |
|----------------------|------------------|
| Bot activity + low source diversity | HIGH inauthenticity signal |
| High growth + low engagement ratio | MODERATE — possible subscriber inflation |
| High forwarding ratio from state media + anonymous admin | MODERATE — possible proxy channel |
| Deletion pattern + coordinated timing | MODERATE — possible narrative testing |
| Single indicator alone | LOW — insufficient for assessment |

## 4. Data Access: Telegram API Capabilities

### 4.1 Bot API (Limited)

| Capability | Available | Notes |
|-----------|-----------|-------|
| Channel metadata | Yes | `getChat` — title, description, member count |
| Message history | **No** | Bots cannot read channel history unless admin |
| Forward metadata | Yes (in received messages) | `forward_from_chat`, `forward_date` |
| Subscriber list | **No** | Privacy restriction |
| Message search | **No** | Not available via Bot API |

### 4.2 MTProto API (Telethon/Pyrogram)

| Capability | Available | Notes |
|-----------|-----------|-------|
| Channel metadata | Yes | Full channel info including creation date (approximate) |
| Message history | **Yes** | `get_messages()` — full history for public channels |
| Forward metadata | **Yes** | Source channel, original date, author (if not anonymous) |
| Subscriber list | **Partial** | Only if user account has admin access |
| Message search | **Yes** | Full-text search within channels |
| View counts | **Yes** | Per-message view counts |
| Deleted message detection | **Partial** | Gap detection via message IDs |

### 4.3 Rate Limits

- MTProto: FloodWait errors (variable 5-60+ seconds)
- Recommended: ~30 requests/second with exponential backoff
- Account-based limits, not IP-based
- See `research/api_rate_limiting.md` for full design

### 4.4 CDDBS Recommendation

**Use MTProto (via Telethon)** for Telegram data collection:
- Provides message history access essential for analysis
- Forward metadata enables forwarding chain reconstruction
- View counts enable engagement analysis
- Requires a user account (phone number) — operational security consideration

## 5. Russian Disinformation on Telegram

### 5.1 Known Russian State Media Channels

| Channel | Subscribers (est.) | Language | Type |
|---------|-------------------|----------|------|
| RT English | ~500K+ | English | Official state media (Tier 1) |
| RT на русском | ~1M+ | Russian | Official state media (Tier 1) |
| Sputnik | ~200K+ | Multi-language | Official state media (Tier 1) |
| TASS | ~500K+ | Russian | State wire service (Tier 1) |
| RIA Novosti | ~1.5M+ | Russian | State wire service (Tier 1) |
| Readovka | ~1M+ | Russian | Pro-Kremlin (Tier 2) |
| WarGonzo | ~1M+ | Russian | War correspondent, pro-Kremlin (Tier 2) |

### 5.2 Observed Propagation Patterns

Based on published research from DFRLab, Stanford IO, and Graphika:

1. **Telegram-first testing**: New narratives appear on anonymous Telegram channels 12-24 hours before appearing on RT/Sputnik Twitter
2. **Cross-platform pipeline**: Telegram channel → screenshot → Twitter post → Facebook group → blog post
3. **Telegram as safe harbor**: After EU/US sanctions on RT (2022), Telegram became the primary distribution channel for RT content in Europe
4. **Channel cloning**: When a channel is flagged, operators create near-identical channels with slightly different names
5. **Comment section manipulation**: Channels with linked discussion groups use coordinated commenters to amplify approved reactions

### 5.3 Telegram Narrative Categories (CDDBS Extension)

All 7 narrative categories from `data/known_narratives.json` propagate on Telegram, but with platform-specific characteristics:

| Category | Telegram-Specific Pattern |
|----------|-------------------------|
| Anti-NATO | Heavy use of military footage/maps in Telegram messages; less text, more visual |
| Anti-EU | Forwarded from European far-right Telegram channels (cross-ideology amplification) |
| Ukraine | Real-time battlefield updates from Telegram war correspondents; rapid forwarding chains |
| Western Hypocrisy | Screenshots of Western news articles with commentary overlaid |
| Global South | Multi-language channels targeting specific regions (Africa, MENA, Latin America) |
| Health Disinfo | Declined on Telegram after COVID peak; residual anti-vaccine content |
| Election Interference | Election-cycle spikes; localized to target country Telegram communities |

## 6. CDDBS Template Adaptation for Telegram

### 6.1 Subject Profile Changes

| Field | Twitter | Telegram Equivalent |
|-------|---------|--------------------|
| `account_handle` | @username | @channel_name or t.me/channel_name |
| `followers` | Follower count | Subscriber count |
| `following` | Following count | N/A (channels don't follow) |
| `total_posts_analyzed` | Tweets | Messages |
| `account_bio` | Bio text | Channel description |
| `verified` | Twitter verification | Telegram verification badge |
| `account_created` | Visible creation date | Not directly exposed; estimate from first message |

### 6.2 New Platform Metadata Fields

```json
"platform_metadata": {
  "telegram_channel_type": "channel",
  "telegram_subscriber_count": 524000,
  "telegram_is_verified": false,
  "telegram_linked_chat": "@rt_english_chat",
  "telegram_invite_link": "t.me/rt_english"
}
```

### 6.3 New Evidence Types

| Type | Schema Key | Description |
|------|-----------|-------------|
| Forwarded message | `forward` | Message forwarded from identified source channel |
| Channel metadata | `channel_meta` | Subscriber count, description, creation date estimate |

## 7. Academic Research Summary

### Key Publications

| Source | Finding | Relevance to CDDBS |
|-------|---------|--------------------|
| **DFRLab** (2022-2025) | Documented extensive Russian Telegram networks operating in Africa and Middle East | Validates multi-language, multi-region Telegram analysis need |
| **Stanford IO** (2023) | Telegram used as coordination layer for cross-platform operations; forwarding chains traceable | Supports forwarding chain analysis as key CDDBS feature |
| **Graphika** (2022-2024) | "Spamouflage" operation expanded to Telegram; used channel networks for Chinese state messaging | Multi-state-actor applicability beyond Russia |
| **NATO StratCom COE** (2023) | Telegram channel networks exhibit measurable coordination patterns detectable through temporal analysis | Validates CIB detection approach via timing correlation |
| **EU DisinfoLab** (2023-2024) | Documented "Doppelganger" operation using Telegram channels to distribute cloned news websites | Validates channel-to-website pipeline analysis |
| **Oxford II** (2022) | Computational propaganda on Telegram grows faster than any other platform | Prioritizes Telegram as urgent CDDBS extension |

### Key Methodological Insights

1. **Forwarding metadata is gold**: Unlike Twitter where retweet attribution can be ambiguous, Telegram forwards explicitly name the source. This makes CDDBS amplification analysis more precise on Telegram.

2. **View counts enable engagement scoring**: Twitter engagement (likes, retweets) requires separate API calls. Telegram view counts are embedded in message data.

3. **Channel admin anonymity is the key challenge**: Attribution is significantly harder on Telegram. CDDBS must rely more on behavioral and network analysis, less on direct account metadata.

4. **Telegram as canary**: New narratives often appear on Telegram before other platforms. Monitoring Telegram channels can provide early warning for emerging disinformation campaigns.

## 8. Conclusions & Recommendations

### For CDDBS v1.3

1. **Add Telegram as first-class platform** in schema, prompt, and quality scorer
2. **Prioritize forwarding chain analysis** as the primary Telegram-specific analytical capability
3. **Use MTProto (Telethon)** for data collection — Bot API is insufficient
4. **Design for admin anonymity** — behavioral/network analysis must compensate for lack of admin metadata
5. **Track cross-platform propagation** — Telegram → Twitter pipeline is a key disinformation vector
6. **Add Telegram-specific behavioral indicators** to both schema and quality scorer
7. **Update known narratives** with Telegram-specific propagation patterns

### Data Access Priority

| Data Type | Priority | Method | Difficulty |
|-----------|----------|--------|------------|
| Message history | P0 | MTProto | Medium (rate limits) |
| Forward metadata | P0 | MTProto (in messages) | Low |
| Channel metadata | P0 | Bot API or MTProto | Low |
| View counts | P1 | MTProto (in messages) | Low |
| Subscriber count | P1 | Bot API | Low |
| Subscriber list | P2 | MTProto (admin only) | High |
| Deleted messages | P2 | Gap detection | Medium |